# 1. Từ kết quả triển khai logistic regression và NN (assignment 2_2 và 4_2), so sánh hai phương pháp.

# 2. Sử dụng thư viện Pytorch để xây dựng mô hình NN phân loại các chữ số viết tay trên MNIST dataset.

# 2.1. Xây dựng mô hình từ Pytorch

Dataset này gồm các chữ số từ 0 đến 9. Train_dataset gồm 60000 mẫu, test_dataset gồm 10000 mẫu, mỗi mẫu là một ảnh đen trắng có kích thước 1x28x28. Đầu ra bằng 1 tương ứng với ảnh chữ số 5 và bằng 0 ứng với ảnh không phải chữ số 5.

In [ ]:
from utils import *
torch.manual_seed(1)
np.random.seed(1)

In [ ]:
train_dataset = datasets.MNIST(root = "dataset/", train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root = "dataset/", train=False, transform=transforms.ToTensor(), download=True)

some_img, some_label = train_dataset[1]
print(len(train_dataset))
print(len(test_dataset))
print(some_img.shape)
plt.imshow(some_img[0,:,:], cmap='binary')
plt.axis('off')
plt.show()

Ở đây ta sẽ xây dựng một mô hình NN sử dụng thư viện Pytorch. Trong file utils.py có phần 'import torch.nn as nn', torch.nn bao gồm nhiều hàm thường được sử dụng mà người dùng có thể gọi ra thay vì phải xây dựng từ đầu, ví dụ như nn.Linear, nn.ReLU, nn.Sigmoid,....

Class nn.Module là một class cơ bản của Pytorch có sẵn hàm forward propagation (đưa x ở đầu vào mô hình và trả đầu ra y), backward propagation (tính toán gradient descent), save_state và load_state (lưu và nạp trạng thái của mô hình),... giúp việc xây dựng các mô hình theo ý muốn dễ dàng hơn.

Ví dụ dưới đây xây dựng mạng NN gồm 2 lớp Linear. Hàm nn.Linear() yêu cầu 2 tham số là in_features và out_features, in_features là số neural ở đầu vào còn out_features là số neural ở đầu ra. Vì mỗi ảnh có kích thước [1,28,28] nên số neural đầu vào lớp đầu tiên là 28x28=784, đầu ra ở lớp cuối là 10 neural ứng với 10 class.

Bài tập 2.1: Hãy hoàn thiện code mô hình NN 2 lớp ở dưới. Layers_dims là một list chứa số neural ở các lớp, vd: [784,20,10]

In [ ]:
class LinearNN_2layer(nn.Module):
    def __init__(self, layers_dims):
        super().__init__()

        self.L1 = nn.Linear(in_features=layers_dims[0], out_features=layers_dims[1])
        self.relu = nn.ReLU(inplace=True) # inplace=True means do not creat new tensor

        # Finish this code

        # End

    def forward(self,x):
        x = self.relu(self.L1(x))
        x = self.sigmoid(self.L2(x))
        return x

Phần dưới sẽ là việc thiết lập DataLoader, loss, optimizer và các hyperparameter cho mô hình. Về module DataLoader và Dataset của Pytorch thì tạm thời chưa yêu cầu các em tìm hiểu sâu, hiểu đơn giản thì đây là công cụ giúp quá trình load data và tính toán trong máy tính nhanh, đơn giản và gọn hơn, đồng thời thực hiện một số quá trình tiền xử lý dữ liệu.

Batch_size, epoch/iteration và optimizer là gì?

In [ ]:
batch_size = 32
num_epochs = 100
lr = 1e-3
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

model_NN2Layer = LinearNN_2layer([784,20,10])
CELoss = nn.CrossEntropyLoss()
optimizer_NN2Layer = optim.SGD(model_NN2Layer.parameters(), lr = lr)

Chạy code ở dưới để huấn luyện mô hình với các hyperparameter đã chọn và quan sát kết quả của mô hình.

In [ ]:
trained_model, NN2Layer_acc_list = trainer(model_NN2Layer, train_loader, test_loader, num_epochs, batch_size, optimizer_NN2Layer, CELoss)
print(f'Linear NN with 2 layers: Train_acc: {NN2Layer_acc_list[-1][0]:.2f}%. Test_acc: {NN2Layer_acc_list[-1][1]:.2f}%.')
plot_curve(NN2Layer_acc_list, curve_type='both')

# 2.2. Tweaking the hyperparameters

Phần này ta sẽ khảo sát sự ảnh hưởng của các hyperparameter đến hoạt động mô hình. Bước đầu tiên là xây dựng một NN có L lớp Linear, trong đó L-1 lớp đầu tiên dùng hàm kích hoạt ReLU và lớp cuối cùng dùng hàm Sigmoid.

model = nn.Sequential() là một chuỗi các hàm được thêm vào khi khởi tạo hoặc bởi model.add_module().
Về lý do sử dụng hàm này, xét trường hợp NN 2 lớp ***INPUT -> LINEAR -> RELU -> LINEAR -> SIGMOID -> OUTPUT***, thay vì phải gọi lần lượt từng hàm L1, ReLU, L2, Sigmoid như trên thì ta có thể xây dựng

    model = nn.Sequential()
    model.add_module('L1',nn.Linear(in_features=2,out_features=10))
    model.add_module('Acti1',nn.ReLU())
    model.add_module('L2',nn.Linear(in_features=10,out_features=1))
    model.add_module('Acti2',nn.Sigmoid())

và sau đó gọi 
    
    output = model(input).

Bài tập 2.2: Xây dựng một mạng NN gồm L lớp Linear: 

Đầu vào ở lớp Linear đầu tiên là 28x28=784, đầu ra ở lớp Linear cuối cùng là 1.

Đầu vào ở lớp Linear thứ l bằng layers_dims[l], đầu ra là layers_dims[l+1].

In [ ]:
class LinearNN(nn.Module):
    def __init__(self, L, layers_dims):
        '''
            L: number of linear layers
            layers_dims: list containing the input size and each layer size, of length (number of layers + 1)
        '''
        super().__init__()
        if len(layers_dims) != L+1:
            raise ValueError(f"Length of layers_dims must be L+1. Expected len(layers_dims)={L+1}, got {len(layers_dims)}")
        self.model = nn.Sequential()

        # Insert code here

        # End code here


    def forward(self,x):
        return self.sigmoid(self.model(x))

Chạy đoạn code dưới đây để kiểm tra hoạt động của mô hình.

Expected results: Train_acc và Test_acc tăng lên khi train.

Bài tập 2.3: Từ mô hình đã hoàn thiện, hãy thay đổi các hyperparameter (batch_size, epoch, L, layers_dims), tổng hợp kết quả và đưa ra nhận xét.

In [ ]:
# Change these
L = 2
layers_dims = [784,50,10]
num_epochs = 100
batch_size = 32
# End

model = LinearNN(L, layers_dims)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)
optimizer = optim.SGD(model.parameters(), lr = lr)

trained_model, acc_list = trainer(model, train_loader, test_loader, num_epochs, batch_size, optimizer, CELoss)
print(f'Linear NN: Train_acc: {acc_list[-1][0]:.2f}%. Test_acc: {acc_list[-1][1]:.2f}%.')
plot_curve(acc_list, curve_type='both')